# CSS Layout Cheat Sheet Generator

This notebook builds a one-page CSS layout cheat sheet automatically: it scrapes a starting article, asks an LLM to find related articles, scrapes those too, and then asks the LLM to distill everything into a tidy markdown cheat sheet.

**The pipeline at a glance:**
1. **Collect links** — scrape the starting article and grab every link on the page.
2. **Filter with the LLM** — ask the model to keep only the links that are genuinely related CSS articles (returned as JSON).
3. **Scrape everything** — fetch the starting article plus each related article and combine the text.
4. **Generate** — ask the LLM to turn all that content into a single markdown cheat sheet.

**To run it:** add your `OPENAI_API_KEY` to a `.env` file in this folder, select the `.venv` kernel (top-right), and run the cells top to bottom. To target a different topic, just change `starting_article_url` in the setup cell.

In [2]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

## Setup and configuration

This cell loads your API key from a `.env` file, picks the model, and connects to OpenAI.

A few things to check:
- You need a `.env` file in this folder containing `OPENAI_API_KEY=sk-...`
- `MODEL` controls which model is used for every call below.
- **`starting_article_url` is the one knob that drives the whole notebook** — change it to point at any plain-HTML article and rerun top to bottom to generate a cheat sheet on a different topic.

In [3]:
# initialize the constants 
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-'):
    print('API key found')
else:
    print('No API key found - please head over to the troubleshooting notebook in this folder to identify & fix!')


MODEL = 'gpt-5-nano'
openai = OpenAI()

# The article to build the cheat sheet from - change this to target a different page
starting_article_url = "https://alistapart.com/article/css-positioning-101/"

API key found


## Step 1 — Collect the links on the page

First we grab every hyperlink on the starting article. This is the raw, unfiltered list straight from the HTML, so it includes navigation, share buttons, and comment anchors — we'll let the LLM clean it up in the next step.

In [ ]:

links = fetch_website_links(starting_article_url)
links

['#content',
 'https://alistapart.com/',
 'https://alistapart.com/',
 'https://alistapart.com',
 'https://alistapart.com/articles/',
 'https://alistapart.com/events/',
 'https://alistapart.com/topics/',
 'https://alistapart.com/about/contribute/',
 'https://alistapart.com/issue/318',
 'https://alistapart.com/author/nstokes/',
 'https://alistapart.com/blog/topic/css/',
 'https://alistapart.com/blog/topic/html/',
 'https://alistapart.com/blog/topic/layout-grids/',
 'https://alistapart.com/article/css-positioning-101/#comments',
 '#section1',
 'https://alistapart.com/article/css-positioning-101/?share=bluesky',
 'https://alistapart.com/article/css-positioning-101/?share=mastodon',
 'https://alistapart.com/article/css-positioning-101/?share=threads',
 'https://www.patreon.com/alistapart',
 'http://www.seminimalist.com/css-positioning-101/',
 'http://kleines-universum.de/html-css/css-position-einmaleins-aus-alistapart/',
 'http://www.clearboth.org/wiki/doku.php?id=ala:css-positioning-101',


## Step 2 — Ask the LLM which links are relevant

A page's raw link list is mostly noise (navigation, share buttons, comments). Here we use a classic "one-shot" technique: a **system prompt** tells the model to return only article links, and shows it the exact JSON shape we want back. We then request `response_format={"type": "json_object"}` so the reply is guaranteed-parseable JSON.

The next few cells build the prompt, preview it, and then make the call that returns the filtered list of articles.

In [18]:
link_system_prompt = """
You are provided with a list of links found in a website about css layout. As such you want to figure out which links might be useful for someone to use to stude more about css.
In this case, you only want to save links that are articles.
You should respond in JSON format as in this example.
{
    "links": [
        {"type": "css layout article", "url": "https://alistapart.com/article/css-positioning-101/"}
    ]
}
"""

In [29]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web link to include a cheat sheet about css layout.
Respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
Limit the returned links to 10.
Only save links that seem to be articles, the relavent links will have the word 'article' in the url in the breadcrumb. Do not save link that are about pages, or about the author, or about the website.
Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [30]:
print(get_links_user_prompt(starting_article_url))


Here is the list of links on the website https://alistapart.com/article/css-positioning-101/ -
Please decide which of these are relevant web link to include a cheat sheet about css layout.
Respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
Limit the returned links to 10.
Only save links that seem to be articles, the relavent links will have the word 'article' in the url in the breadcrumb. Do not save link that are about pages, or about the author, or about the website.
Links (some might be relative links):

#content
https://alistapart.com/
https://alistapart.com/
https://alistapart.com
https://alistapart.com/articles/
https://alistapart.com/events/
https://alistapart.com/topics/
https://alistapart.com/about/contribute/
https://alistapart.com/issue/318
https://alistapart.com/author/nstokes/
https://alistapart.com/blog/topic/css/
https://alistapart.com/blog/topic/html/
https://alistapart.com/blog/topic/layout-grids/
https://alistapart.

In [31]:
def select_relavent_links(url):
    response = openai.chat.completions.create(
        model= MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [32]:
select_relavent_links(starting_article_url)

{'links': [{'type': 'css layout article',
   'url': 'https://alistapart.com/article/css-positioning-101/?share=bluesky'},
  {'type': 'css layout article',
   'url': 'https://alistapart.com/article/css-positioning-101/?share=mastodon'},
  {'type': 'css layout article',
   'url': 'https://alistapart.com/article/css-positioning-101/?share=threads'},
  {'type': 'css layout article',
   'url': 'https://alistapart.com/article/css-floats-101/'},
  {'type': 'css layout article',
   'url': 'https://alistapart.com/article/mobile-first-css-is-it-time-for-a-rethink/'},
  {'type': 'css layout article',
   'url': 'https://alistapart.com/article/breaking-out-of-the-box/'},
  {'type': 'css layout article',
   'url': 'https://alistapart.com/article/designed-for-a-dead-language/'},
  {'type': 'css layout article',
   'url': 'https://alistapart.com/article/good-designers-bad-websites-a-proposal/'},
  {'type': 'css layout article',
   'url': 'https://alistapart.com/article/design-for-amiability-lessons-fr

## Step 3 — Scrape the article and all the chosen pages

Now we fetch the full text of the starting article plus each related article the LLM picked, and stitch it all into one big string. This combined text becomes the raw material for the cheat sheet.

Note: each page is truncated to 2,000 characters by the scraper (see `scraper.py`), which keeps the prompt small and your API costs low.

In [34]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relavent_links(url)
    result = f"## CSS Layout Cheat Sheet:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result    

In [35]:
relative_page_date = fetch_page_and_all_relevant_links(starting_article_url)
print(relative_page_date)

## CSS Layout Cheat Sheet:

CSS Positioning 101 – A List Apart

Skip to content
A List Apart
For people who make websites
Primary Menu
Home
Articles
Events
Topics
Write for Us
Search for:
Issue №
318
CSS Positioning 101
by
Noah Stokes
November 16, 2010
Published in
CSS
,
HTML
,
Layout & Grids
If you’re a front end developer or a designer who likes to code, CSS-based layouts are at the very core of your work. In what might be a refresher for some, or even an “a-ha!” for others, let’s look at the CSS
position
property to see how we can use it to create standards-compliant, table-free CSS layouts.
Article Continues Below
Article Continues Below
66 Comments
Share this:
#section1
Share on Bluesky (Opens in new window)
Bluesky
Share on Mastodon (Opens in new window)
Mastodon
Share on Threads (Opens in new window)
Threads
Become a patron
Translations
Chinese
German
Korean
Polish
Ready to advance in content strategy, UX/UI, data communication, or learning design? Explore Northwestern’s online 

## Step 4 — Generate the cheat sheet

Finally, we hand all of the gathered content to the LLM with a system prompt that asks it to act like a front-end developer and distill everything into a concise markdown cheat sheet (with code blocks for the CSS examples).

`create_cheat_sheet(...)` both displays the result and returns it as a string, so you can capture it in a variable if you want to save or export it later:

```python
cheat_sheet_markdown = create_cheat_sheet(starting_article_url)
```

In [36]:
cheat_sheet_system_prompt = """
You are an front end developer that analyzes the contents of several relevant pages from a CSS Layout website.
You are creating a cheat sheet for other front end developers to use to prompt their memory about CSS layout.
Respond in markdown using code blocks for the CSS layout examples.
"""

In [37]:
def get_cheat_sheet_user_prompt(url):
    user_prompt = f"""
Here is the contents of the page {url} -
Please create a cheat sheet for other front end developers to use to prompt their memory about CSS layout.
Respond in markdown using code blocks for the CSS layout examples.
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    return user_prompt

In [39]:
def create_cheat_sheet(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": cheat_sheet_system_prompt},
            {"role": "user", "content": get_cheat_sheet_user_prompt(url)}
        ],
        response_format={"type": "text"}
    )
    result = response.choices[0].message.content
    display(Markdown(result))
    return result



In [40]:
create_cheat_sheet(starting_article_url)

## CSS Layout Cheat Sheet: Positioning (based on CSS Positioning 101)

Key ideas from the article:
- The CSS position property has five main values: static, relative, absolute, fixed, and inherit.
- How an element is positioned determines how it participates in the normal flow and how it is laid out relative to its containing block.
- Absolute positioning is based on the nearest positioned ancestor (a parent with a position other than static). If none exists, it is relative to the initial containing block (often the viewport).
- Relative positioning shifts an element from its normal position but preserves its original space in the flow.
- Fixed positioning anchors elements to the viewport and does not move with scrolling.
- Inherit makes the element take the same position value as its parent.

Quick reference (one-liners)
- static: default flow, no offset
- relative: offset from normal position; still takes up space
- absolute: removed from normal flow; positioned relative to nearest positioned ancestor
- fixed: positioned relative to viewport; does not scroll with page
- inherit: same positioning as parent

Common tips
- Use a positioned ancestor (e.g., .container { position: relative; }) when you want absolutely positioned children to be contained.
- Use transform: translate(...) for smoother, more predictable centering offsets with relative/absolute.
- Combine z-index with positioning to manage stacking context.

Code examples

1) Static (default flow)
- No special positioning needed; elements participate in normal flow.

```html
<div class="static-layout">
  <div class="box a">A</div>
  <div class="box b">B</div>
</div>
```

```css
.static-layout {
  /* normal flow: blocks stack vertically by default */
}
.box {
  width: 100px;
  height: 60px;
  background: #4a90e2;
  color: white;
  display: inline-block;
  margin: 8px;
}
```

2) Relative positioning (offset from normal position)
- The element remains in the document flow, but is shifted.

```html
<div class="relative-layout">
  <div class="box r1">Rel 1</div>
  <div class="box r2" style="margin-left: 16px;">Rel 2</div>
</div>
```

```css
.relative-layout {
  /* no special container needed */
}
.box.r1 {
  position: relative;
  top: 6px;    /* move down 6px from its normal position */
  left: 12px;  /* move right 12px from its normal position */
  background: #e67e22;
}
.box.r2 {
  position: relative;
  top: -4px;   /* move up a bit relative to normal position */
  background: #16a085;
}
```

3) Absolute positioning within a containing block
- Child is positioned relative to the nearest positioned ancestor (often a wrapper with position: relative).

```html
<div class="abs-container">
  <div class="box abs-child">Absolute</div>
</div>
```

```css
.abs-container {
  position: relative; /* anchor for the absolutely positioned child */
  width: 300px;
  height: 150px;
  background: #f0f0f0;
  border: 1px solid #ccc;
}
.box.abs-child {
  position: absolute;
  top: 20px;
  right: 20px;
  width: 120px;
  height: 60px;
  background: #d35400;
  color: white;
  display: flex;
  align-items: center;
  justify-content: center;
}
```

4) Fixed positioning (viewport-anchored)
- Element stays fixed in the viewport, independent of page scroll.

```html
<div class="fixed-bar">Fixed header</div>
<div class="content">
  <p>Scroll to see the effect of a fixed element staying in place.</p>
  <!-- lots of content to enable scrolling -->
  <p style="height: 2000px;"></p>
</div>
```

```css
.fixed-bar {
  position: fixed;
  top: 0;
  left: 0;
  right: 0;
  height: 54px;
  background: #2c3e50;
  color: white;
  display: flex;
  align-items: center;
  justify-content: center;
  z-index: 1000;
}
.content {
  padding-top: 70px; /* prevent content from being hidden under the fixed bar */
}
```

5) Inherit positioning (child takes the same position type as its parent)
- The child inherits the position value from its parent.

```html
<div class="parent">
  Parent (position: relative)
  <div class="child-inherit">Child (position: inherit)</div>
</div>
```

```css
.parent {
  position: relative;
  width: 320px;
  height: 140px;
  background: #f9f9f9;
  border: 1px solid #ccc;
  padding: 12px;
}
.child-inherit {
  position: inherit; /* same as parent's position: relative */
  top: 10px;
  left: 8px;
  width: 140px;
  height: 50px;
  background: rgba(52, 152, 219, 0.9);
  color: white;
  display: flex;
  align-items: center;
  justify-content: center;
}
```

6) Centering an element with absolute positioning (classic pattern)
- Use top/left 50% and translate to perfectly center.

```html
<div class="viewport">
  <div class="centered">Centered</div>
</div>
```

```css
.viewport {
  position: relative;
  width: 400px;
  height: 240px;
  border: 1px solid #ccc;
}
.centered {
  position: absolute;
  top: 50%;
  left: 50%;
  transform: translate(-50%, -50%);
  padding: 8px 16px;
  background: #8e44ad;
  color: white;
}
```

7) Overlay example (absolute within a container, common for modal/tooltip)
- Overlay sits on top of content using z-index.

```html
<div class="page">
  <div class="content">Main content here...</div>
  <div class="overlay" aria-label="Modal overlay">
    Overlay content
  </div>
</div>
```

```css
.page { position: relative; }
.overlay {
  position: absolute;
  top: 0; left: 0; right: 0; bottom: 0;
  background: rgba(0, 0, 0, 0.5);
  color: white;
  display: flex;
  align-items: center;
  justify-content: center;
  z-index: 999;
}
```

8) Practical notes and pitfalls
- Absolute elements are removed from normal flow; they don’t take up space unless you compensate with layout.
- Relative elements still occupy their original space; the offset is visual only.
- Fixed elements stay in place during scrolling; good for headers, toolbars, or floating widgets.
- Always consider the containing block for absolute positioning; use a positioned ancestor to control where the child is placed.
- Use z-index to control stacking when elements overlap; remember that z-index only applies to positioned elements (position other than static).

If you’d like, I can tailor this cheat sheet to a specific project pattern (e.g., grid-like layouts with positioned cards, tooltips, modals) or expand any section with more real-world examples.

'## CSS Layout Cheat Sheet: Positioning (based on CSS Positioning 101)\n\nKey ideas from the article:\n- The CSS position property has five main values: static, relative, absolute, fixed, and inherit.\n- How an element is positioned determines how it participates in the normal flow and how it is laid out relative to its containing block.\n- Absolute positioning is based on the nearest positioned ancestor (a parent with a position other than static). If none exists, it is relative to the initial containing block (often the viewport).\n- Relative positioning shifts an element from its normal position but preserves its original space in the flow.\n- Fixed positioning anchors elements to the viewport and does not move with scrolling.\n- Inherit makes the element take the same position value as its parent.\n\nQuick reference (one-liners)\n- static: default flow, no offset\n- relative: offset from normal position; still takes up space\n- absolute: removed from normal flow; positioned relativ

## Exporting your cheat sheet to PDF

You've got the cheat sheet as markdown. Here are two ways to turn it into a PDF.

### Option A — No code (easiest)
1. Run the cell that displays the cheat sheet so it renders.
2. In your browser/Jupyter, choose **File → Print** (or Ctrl/Cmd+P).
3. Set the destination to **"Save as PDF"** and save.

### Option B — Generate the PDF in Python
Run the code cell below. It converts the markdown to HTML, then to a PDF file called `cheat_sheet.pdf` in this folder.

First install the two packages it needs (run once): `pip install markdown weasyprint`

This assumes you captured the cheat sheet into a variable, for example: `cheat_sheet_markdown = create_cheat_sheet(starting_article_url)`


In [ ]:
# Export the cheat sheet markdown to a PDF.
# Run once if needed:  pip install markdown weasyprint
import markdown
from weasyprint import HTML

# Capture the cheat sheet markdown. If you already stored it earlier, you can skip this line.
cheat_sheet_markdown = create_cheat_sheet(starting_article_url)

html_body = markdown.markdown(
    cheat_sheet_markdown,
    extensions=["fenced_code", "codehilite", "tables"],
)

# A little CSS so code blocks and the page look reasonable in print
styled_html = f"""
<html>
  <head>
    <style>
      body {{ font-family: -apple-system, Arial, sans-serif; line-height: 1.5; margin: 2rem; }}
      pre {{ background: #f4f4f4; padding: 1rem; border-radius: 6px; white-space: pre-wrap; }}
      code {{ font-family: Menlo, Consolas, monospace; }}
      h1, h2, h3 {{ color: #222; }}
    </style>
  </head>
  <body>{html_body}</body>
</html>
"""

HTML(string=styled_html).write_pdf("cheat_sheet.pdf")
print("Saved cheat_sheet.pdf")